In [1]:
# DEPENDENCIAS
!pip install torchviz
!pip install torchsummary

In [2]:
# LIBRERIAS
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import mean_squared_error, mean_absolute_error
from torchsummary import summary
from torchviz import make_dot
import zipfile
import shutil

In [3]:
# EXTRACCION DE DATOS
# Path zip
zip_path = "/content/Categorias.zip"
extract_path = "/content"

# Crear carpeta destino si no existe
os.makedirs(extract_path, exist_ok=True)

# Descomprimir
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Archivos extraídos en:", extract_path)

Archivos extraídos en: /content


In [4]:
# DATASET
class train_dataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        data_path = os.path.join(self.root_dir, self.annotations.iloc[index, 0])
        data = pd.read_csv(data_path).values.astype(np.float32)
        data = torch.tensor(data).unsqueeze(0)  # [1, 24, 43]
        y_label = torch.tensor(int(self.annotations.iloc[index, 1]), dtype=torch.long)

        if self.transform:
            data = self.transform(data)

        return (data, y_label)

In [5]:
# MODELO
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc1 = nn.Linear(24*44 + 13, 800)
        self.fc2 = nn.Linear(800, 400)
        self.fc3 = nn.Linear(400, 200)
        self.fc_mu = nn.Linear(200, latent_dim)
        self.fc_logvar = nn.Linear(200, latent_dim)

    def forward(self, x, y):
        inputs = torch.cat([x, y], dim=1)
        h = F.relu(self.fc1(inputs))
        h = F.relu(self.fc2(h))
        h = F.relu(self.fc3(h))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim + 13, 200)
        self.fc2 = nn.Linear(200, 400)
        self.fc3 = nn.Linear(400, 800)
        self.fc4 = nn.Linear(800, 24*44)

    def forward(self, z, y):
        inputs = torch.cat([z, y], dim=1)
        h = F.relu(self.fc1(inputs))
        h = F.relu(self.fc2(h))
        h = F.relu(self.fc3(h))
        out = torch.sigmoid(self.fc4(h))
        return out

class CVAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x, y):
        mu, logvar = self.encoder(x, y)
        z = self.reparameterize(mu, logvar)
        out = self.decoder(z, y)
        return out, mu, logvar

In [6]:
# FUNCION DE PERDIDA
# reconstrucción + KL divergence
def loss_function(recon_x, x, mu, logvar):
    BCE = F.binary_cross_entropy(recon_x, x, reduction="sum")
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD

In [7]:
# METRICAS ADICIONALES
def reconstruction_metrics(recon_x, x):
    recon_x = recon_x.detach().cpu().numpy()
    x = x.detach().cpu().numpy()
    mse = mean_squared_error(x.flatten(), recon_x.flatten())
    mae = mean_absolute_error(x.flatten(), recon_x.flatten())
    psnr = 20 * np.log10(1.0 / np.sqrt(mse + 1e-10))  # datos entre [0,1]
    return mse, mae, psnr

def latent_metrics(mu, logvar, z):
    z_mean_norm = z.norm(dim=1).mean().item()
    z_var_mean = logvar.exp().mean().item()
    return z_mean_norm, z_var_mean

In [8]:
# PARAMETROS
batch_size = 64
latent_dim = 20
epochs = 200
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [9]:
# CARGAR DATOS
dataset = train_dataset(csv_file="/content/df_grupos.csv",
                        root_dir="/content/unificado")
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Mostrar un dato de ejemplo
data, label = dataset[100]
print("Shape del dato:", data.shape)
print("Etiqueta:", label)

Shape del dato: torch.Size([1, 24, 44])
Etiqueta: tensor(4)


In [10]:
# INTANCIA DEL MODELO
model = CVAE(latent_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [11]:
# CODIFICADOR DE CONDICIONES
# Función para convertir etiquetas a one-hot
def to_one_hot(labels, num_classes=13):
    return F.one_hot(labels, num_classes).float()

In [12]:
# LOOP DE ENTRENAMIENTO
model.train()
for epoch in range(epochs):
    train_loss = 0
    mse_total, mae_total, psnr_total = 0, 0, 0
    z_norm_total, z_var_total = 0, 0

    for batch_idx, (data, labels) in enumerate(train_loader):
        data = data.view(-1, 24*44).to(device)
        labels_one_hot = to_one_hot(labels - 1, num_classes=13).to(device)

        optimizer.zero_grad()
        recon_batch, mu, logvar = model(data, labels_one_hot)
        loss = loss_function(recon_batch, data, mu, logvar)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()

        # === Métricas ===
        mse, mae, psnr = reconstruction_metrics(recon_batch, data)
        z = model.reparameterize(mu, logvar)
        z_norm, z_var = latent_metrics(mu, logvar, z)

        mse_total += mse
        mae_total += mae
        psnr_total += psnr
        z_norm_total += z_norm
        z_var_total += z_var

    n_batches = len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} "
          f"Loss: {train_loss/len(train_loader.dataset):.4f} | "
          f"MSE: {mse_total/n_batches:.6f} | "
          f"MAE: {mae_total/n_batches:.6f} | "
          f"PSNR: {psnr_total/n_batches:.2f} dB | "
          f"Z-norm: {z_norm_total/n_batches:.4f} | "
          f"Z-var: {z_var_total/n_batches:.6f}")

Epoch 1/200 Loss: 680.7467 | MSE: 0.088511 | MAE: 0.250646 | PSNR: 10.73 dB | Z-norm: 4.9851 | Z-var: 1.223054
Epoch 2/200 Loss: 573.5312 | MSE: 0.046839 | MAE: 0.162392 | PSNR: 13.33 dB | Z-norm: 5.5293 | Z-var: 1.433085
Epoch 3/200 Loss: 545.1905 | MSE: 0.044208 | MAE: 0.155464 | PSNR: 13.55 dB | Z-norm: 4.5042 | Z-var: 1.040441
Epoch 4/200 Loss: 537.8343 | MSE: 0.040303 | MAE: 0.139679 | PSNR: 13.96 dB | Z-norm: 4.4522 | Z-var: 1.017133
Epoch 5/200 Loss: 534.8587 | MSE: 0.041669 | MAE: 0.145684 | PSNR: 13.81 dB | Z-norm: 4.4659 | Z-var: 0.996104
Epoch 6/200 Loss: 532.5723 | MSE: 0.039866 | MAE: 0.141046 | PSNR: 14.00 dB | Z-norm: 4.3661 | Z-var: 0.993182
Epoch 7/200 Loss: 531.6765 | MSE: 0.040434 | MAE: 0.140981 | PSNR: 13.94 dB | Z-norm: 4.3413 | Z-var: 0.956822
Epoch 8/200 Loss: 529.9875 | MSE: 0.038816 | MAE: 0.138815 | PSNR: 14.11 dB | Z-norm: 4.2384 | Z-var: 0.948058
Epoch 9/200 Loss: 528.4218 | MSE: 0.039130 | MAE: 0.138971 | PSNR: 14.08 dB | Z-norm: 4.1077 | Z-var: 0.857792
E

In [13]:
# EXPORTAR EL MODELO
torch.save(model.state_dict(), "modelo_cvae.pth")

In [14]:
# EXPORTAR ARQUITECTURA UTILIZADA
latent_dim = 20
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CVAE(latent_dim).to(device)

# La entrada del modelo es una tupla: (Matriz aplanada, etiqueta one-hot)
# Imagen: 24x44 → 1056, Etiqueta: 13
summary(model, [(24*44,), (13,)])



----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                  [-1, 800]         856,000
            Linear-2                  [-1, 400]         320,400
            Linear-3                  [-1, 200]          80,200
            Linear-4                   [-1, 20]           4,020
            Linear-5                   [-1, 20]           4,020
           Encoder-6       [[-1, 20], [-1, 20]]               0
            Linear-7                  [-1, 200]           6,800
            Linear-8                  [-1, 400]          80,400
            Linear-9                  [-1, 800]         320,800
           Linear-10                 [-1, 1056]         845,856
          Decoder-11                 [-1, 1056]               0
Total params: 2,518,496
Trainable params: 2,518,496
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.05
Forw

In [15]:
# ARQUITECTURA GRAFICA
# modelo CVAE ya  definido
latent_dim = 20
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CVAE(latent_dim).to(device)

# Crear un ejemplo de entrada (batch_size=1)
x = torch.randn(1, 24*44).to(device)  # Data aplanada
y = torch.randn(1, 13).to(device)     # etiqueta one-hot

# Pasar por el modelo
out, mu, logvar = model(x, y)

# Generar el grafo
dot = make_dot(out, params=dict(model.named_parameters()))
dot.format = 'png'
dot.render('Arquitectura_CVAE')  # guardará Arquitectura_CVAE.png

'Arquitectura_CVAE.png'

In [16]:
# GENERACION DE DATOS
def generar_dato_mes(mes):
  # GENERACION CONDICIONAL
  latent_dim = 20
  # CARGAR EL MODELO
  model = CVAE(latent_dim)
  model.load_state_dict(torch.load("modelo_cvae.pth"))
  model.eval()
  z = torch.randn(1, latent_dim).to(device)  # vector latente aleatorio
  digit = mes  # etiqueta deseada en formato one-hot
  y = F.one_hot(torch.tensor([digit]), num_classes=13).float().to(device)

  # Pasar z y y al decoder para generar la imagen
  with torch.no_grad():
      generated_data = model.decoder(z, y)  # salida tamaño (1, 24x43)
      generated_data = generated_data.view(24, 44).cpu()  # darle forma de 24x43

  # mover a CPU y convertir a numpy
  data_numpy = generated_data.cpu().numpy()

  # crear DataFrame directamente con la forma 24x43
  df_generated = pd.DataFrame(data_numpy)
  return df_generated

In [17]:
def generar_datos_mes(ruta_directorio, n_archivos, tipo_mes):
  # Crear el directorio si no existe
  os.makedirs(ruta_directorio, exist_ok=True)
  for i in range(n_archivos):
      # Crear DataFrame
      df_gen_aux = generar_dato_mes(tipo_mes)
      # Nombre del archivo
      filename = os.path.join(ruta_directorio, f"df_{i}.csv")

      # Guardar en CSV
      df_gen_aux.to_csv(filename, index=False)

  print(f"Se han generado y guardado {n_archivos} CSVs en la carpeta: {ruta_directorio}")


In [18]:
# GENERAR CADA DIA DEL MES
generar_datos_mes('gen_enero_2020', 31, 0)
generar_datos_mes('gen_febrero_2020', 28, 1)
generar_datos_mes('gen_marzo_2020', 31, 2)
generar_datos_mes('gen_abril_2020', 30, 3)
generar_datos_mes('gen_mayo_2020', 31, 4)
generar_datos_mes('gen_junio_2020', 30, 5)
generar_datos_mes('gen_julio_2020', 31, 6)
generar_datos_mes('gen_agosto_2020', 31, 7)
generar_datos_mes('gen_septiembre_2020', 30, 8)
generar_datos_mes('gen_octubre_2020', 31, 9)
generar_datos_mes('gen_noviembre_2020', 30, 10)
generar_datos_mes('gen_diciembre_2020', 31, 11)
generar_datos_mes('gen_enero_2021', 31, 12)

Se han generado y guardado 31 CSVs en la carpeta: gen_enero_2020
Se han generado y guardado 28 CSVs en la carpeta: gen_febrero_2020
Se han generado y guardado 31 CSVs en la carpeta: gen_marzo_2020
Se han generado y guardado 30 CSVs en la carpeta: gen_abril_2020
Se han generado y guardado 31 CSVs en la carpeta: gen_mayo_2020
Se han generado y guardado 30 CSVs en la carpeta: gen_junio_2020
Se han generado y guardado 31 CSVs en la carpeta: gen_julio_2020
Se han generado y guardado 31 CSVs en la carpeta: gen_agosto_2020
Se han generado y guardado 30 CSVs en la carpeta: gen_septiembre_2020
Se han generado y guardado 31 CSVs en la carpeta: gen_octubre_2020
Se han generado y guardado 30 CSVs en la carpeta: gen_noviembre_2020
Se han generado y guardado 31 CSVs en la carpeta: gen_diciembre_2020
Se han generado y guardado 31 CSVs en la carpeta: gen_enero_2021


In [19]:
# Crear carpeta "Resultados_Modelo"
os.makedirs("Resultados_Modelo", exist_ok=True)

# Mover a "Resultados_Modelo" las carpetas creadas
shutil.move("gen_enero_2020", "Resultados_Modelo/")
shutil.move("gen_febrero_2020", "Resultados_Modelo/")
shutil.move("gen_marzo_2020", "Resultados_Modelo/")
shutil.move("gen_abril_2020", "Resultados_Modelo/")
shutil.move("gen_mayo_2020", "Resultados_Modelo/")
shutil.move("gen_junio_2020", "Resultados_Modelo/")
shutil.move("gen_julio_2020", "Resultados_Modelo/")
shutil.move("gen_agosto_2020", "Resultados_Modelo/")
shutil.move("gen_septiembre_2020", "Resultados_Modelo/")
shutil.move("gen_octubre_2020", "Resultados_Modelo/")
shutil.move("gen_noviembre_2020", "Resultados_Modelo/")
shutil.move("gen_diciembre_2020", "Resultados_Modelo/")
shutil.move("gen_enero_2021", "Resultados_Modelo/")

# Mover el modelo a la carpeta Resultados_Modelo
shutil.move("/content/modelo_cvae.pth", "Resultados_Modelo/")
shutil.move("/content/Arquitectura_CVAE.png", "Resultados_Modelo/")

# Comprimir la carpeta "Categorias" en formato zip
shutil.make_archive("Resultados_Modelo", 'zip', "Resultados_Modelo")

'/content/Resultados_Modelo.zip'